# Stage 6C — Governed Evidence Extraction

This notebook materializes governed, source-located evidence extracted from the controlled Stage 6B document registry.

Three evidence classes remain separate throughout the stage:

1. documented strategy actions;
2. observable company-result observations; and
3. company-reported attribution claims.

Stage 6C does not infer that a strategy caused a result, does not convert company attribution into independent evidence, does not standardize incompatible measures onto one scale, does not rank the focal groups, does not construct a composite score, and does not prepare a report or README.


## Environment Setup

Import the required libraries, lock the authoritative Stage 6B commit and eight canonical design/acquisition inputs, and define deterministic runtime paths.


In [1]:
from __future__ import annotations

import hashlib
import os
import shutil
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
INPUT_COMMIT = "6c841db591183584ecf7db990322281823125691"
RETRIEVAL_DATE = "2026-08-21"

INPUT_LOCKS = {
    "metadata/stage6b_input_lock.csv":
        "87575313eab43fec7b4f289495fc002a42a3e7ab5b36ef089f65e810e8f095e4",
    "metadata/stage6b_document_registry.csv":
        "3b872109b5f721a438b9c62bf7c400b658db25900fee5b7607e5684f72abcebd",
    "metadata/stage6b_target_coverage.csv":
        "19e315ef8d36899740e27cb65a2c85cdff6ad2c3a4a6f4147d74633387904a3b",
    "metadata/stage6b_acquisition_exceptions.csv":
        "580fbb95336df70fde63ae4ac9f7123b130f0cf0ea3e946a71ce47c4354f28a1",
    "metadata/stage6b_acquisition_validation.csv":
        "3aba60ef0fcfcadef169cd542919488c873a735dd9f946bdfe93f15f63279308",
    "metadata/stage5_strategy_taxonomy.csv":
        "65c5ffd83c847b9f93c64a4bafd8ca6bf27abe26ee5ea2acd2ff69e07862c684",
    "metadata/stage5_outcome_taxonomy.csv":
        "32a3b729ea1e903a24a30e5927e47e909745e527c42281dd0997a7919320cc03",
    "metadata/stage5_attribution_evidence_rules.csv":
        "a2afb7e170f71d6bb6937cfc41c3e84c027c4e2df82525096a2cf776215006a5",
}

OUTPUT_ROOT = Path(
    os.environ.get("FMCG_STAGE6C_OUTPUT_ROOT", "/content/fmcg_stage6c_outputs")
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

print(f"Locked Stage 6B commit: {INPUT_COMMIT}")
print(f"Required governed inputs: {len(INPUT_LOCKS)}")
print(f"Output root: {OUTPUT_ROOT}")


Locked Stage 6B commit: 6c841db591183584ecf7db990322281823125691
Required governed inputs: 8
Output root: /content/fmcg_stage6c_outputs


## Locked Input Retrieval

Retrieve only the eight governed Stage 5/6B inputs from the authoritative Stage 6B commit. Colab uses the `GITHUB_TOKEN` secret; a local validation root can be supplied through `FMCG_STAGE6C_INPUT_ROOT`.


In [2]:
configured_root = os.environ.get("FMCG_STAGE6C_INPUT_ROOT")

if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Run in Google Colab or set FMCG_STAGE6C_INPUT_ROOT for local validation."
        ) from exc

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError(
            "Colab Secret GITHUB_TOKEN is unavailable or access has not been granted."
        )

    INPUT_ROOT = Path("/content/fmcg_stage6c_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for relative_path in INPUT_LOCKS:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        url = (
            f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}"
            f"/contents/{encoded_path}?ref={INPUT_COMMIT}"
        )
        request = urllib.request.Request(
            url,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github.raw+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "fmcg-stage6c-colab",
            },
        )

        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)

        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"GitHub input retrieval failed for {relative_path} "
                f"with HTTP {exc.code}."
            ) from exc

    del github_token
    input_mode = "locked_github_commit"

missing = [
    relative_path
    for relative_path in INPUT_LOCKS
    if not (INPUT_ROOT / relative_path).exists()
]

if missing:
    raise FileNotFoundError(f"Missing governed Stage 6C inputs: {missing}")

print(f"Input mode: {input_mode}")
print(f"Required files found: {len(INPUT_LOCKS)}/{len(INPUT_LOCKS)}")


Input mode: locked_github_commit
Required files found: 8/8


## Input Integrity and Frozen Taxonomies

Verify all inherited artifacts against their locked SHA-256 values, confirm the Stage 6B acquisition gate, and load the frozen strategy, outcome, and attribution taxonomies before any extracted evidence is materialized.


In [4]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


lock_rows = []

for relative_path, expected_sha256 in INPUT_LOCKS.items():
    actual_sha256 = sha256_file(INPUT_ROOT / relative_path)
    lock_rows.append(
        {
            "file_path": relative_path,
            "expected_sha256": expected_sha256,
            "actual_sha256": actual_sha256,
            "hash_match": actual_sha256 == expected_sha256,
            "locked_repository_commit": INPUT_COMMIT,
        }
    )

stage6c_input_lock = pd.DataFrame(lock_rows)

if not stage6c_input_lock["hash_match"].all():
    raise RuntimeError("One or more Stage 6C input checksums failed.")

stage6b_documents = pd.read_csv(
    INPUT_ROOT / "metadata/stage6b_document_registry.csv",
    dtype=str,
    keep_default_na=False,
)

stage6b_coverage = pd.read_csv(
    INPUT_ROOT / "metadata/stage6b_target_coverage.csv",
    dtype=str,
    keep_default_na=False,
)

stage6b_exceptions = pd.read_csv(
    INPUT_ROOT / "metadata/stage6b_acquisition_exceptions.csv",
    dtype=str,
    keep_default_na=False,
)

stage6b_validation = pd.read_csv(
    INPUT_ROOT / "metadata/stage6b_acquisition_validation.csv",
    dtype=str,
    keep_default_na=False,
)

strategy_taxonomy = pd.read_csv(
    INPUT_ROOT / "metadata/stage5_strategy_taxonomy.csv",
    dtype=str,
    keep_default_na=False,
)

outcome_taxonomy = pd.read_csv(
    INPUT_ROOT / "metadata/stage5_outcome_taxonomy.csv",
    dtype=str,
    keep_default_na=False,
)

attribution_rules = pd.read_csv(
    INPUT_ROOT / "metadata/stage5_attribution_evidence_rules.csv",
    dtype=str,
    keep_default_na=False,
)


# ------------------------------------------------------------
# Confirm inherited Stage 6B gate.
# ------------------------------------------------------------

final_stage6b = stage6b_validation.loc[
    stage6b_validation["check_id"] == "S6B033"
].iloc[0]

if (
    final_stage6b["result"] != "PASS_WITH_CAVEAT"
    or final_stage6b["status"] != "passed_with_caveat"
):
    raise RuntimeError(
        "Stage 6B final gate is not the expected "
        "PASS_WITH_CAVEAT status."
    )


# ------------------------------------------------------------
# Confirm inherited document registry.
# ------------------------------------------------------------

if len(stage6b_documents) != 56:
    raise RuntimeError(
        "Expected 56 Stage 6B document references, "
        f"found {len(stage6b_documents)}."
    )

if stage6b_documents["document_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate document IDs found in the Stage 6B registry."
    )


# ------------------------------------------------------------
# Validate frozen Stage 5 taxonomies.
#
# Important:
# STR01–STR09 are stored in strategy_id.
# ATTR01–ATTR09 are stored in attribution_rule_id.
# ------------------------------------------------------------

expected_strategy_ids = {
    f"STR{i:02d}" for i in range(1, 10)
}

actual_strategy_ids = set(
    strategy_taxonomy["strategy_id"]
)

if actual_strategy_ids != expected_strategy_ids:
    raise RuntimeError(
        "Frozen strategy taxonomy is incomplete. "
        f"Expected {sorted(expected_strategy_ids)}, "
        f"found {sorted(actual_strategy_ids)}."
    )


expected_outcome_ids = {
    f"OUT{i:02d}" for i in range(1, 19)
}

actual_outcome_ids = set(
    outcome_taxonomy["outcome_id"]
)

if actual_outcome_ids != expected_outcome_ids:
    raise RuntimeError(
        "Frozen outcome taxonomy is incomplete. "
        f"Expected {sorted(expected_outcome_ids)}, "
        f"found {sorted(actual_outcome_ids)}."
    )


expected_attribution_rule_ids = {
    f"ATTR{i:02d}" for i in range(1, 10)
}

actual_attribution_rule_ids = set(
    attribution_rules["attribution_rule_id"]
)

if actual_attribution_rule_ids != expected_attribution_rule_ids:
    raise RuntimeError(
        "Frozen attribution taxonomy is incomplete. "
        f"Expected {sorted(expected_attribution_rule_ids)}, "
        f"found {sorted(actual_attribution_rule_ids)}."
    )


# ------------------------------------------------------------
# Controlled lookup registries used by subsequent cells.
# ------------------------------------------------------------

document_lookup = (
    stage6b_documents
    .set_index("document_id")
    .to_dict("index")
)

valid_document_ids = set(
    stage6b_documents["document_id"]
)

# These variables intentionally contain the frozen stable IDs
# used by subsequent Stage 6C extraction cells.
valid_strategy_codes = set(
    strategy_taxonomy["strategy_id"]
)

valid_outcome_ids = set(
    outcome_taxonomy["outcome_id"]
)

valid_attribution_classes = set(
    attribution_rules["attribution_rule_id"]
)


def resolved_source_url(document_id: str) -> str:
    if document_id not in document_lookup:
        raise KeyError(
            f"Unknown Stage 6B document ID: {document_id}"
        )

    record = document_lookup[document_id]

    direct_file_url = record["direct_file_url"].strip()

    if direct_file_url:
        return direct_file_url

    return record["source_url"]


print(
    "Input checksums passed: "
    f"{stage6c_input_lock['hash_match'].sum()}/"
    f"{len(stage6c_input_lock)}"
)

print(
    "Stage 6B gate: "
    f"{final_stage6b['result']} / "
    f"{final_stage6b['status']}"
)

print(
    f"Controlled documents available: "
    f"{len(stage6b_documents)}"
)

print(
    f"Frozen strategy IDs: "
    f"{len(valid_strategy_codes)}"
)

print(
    f"Frozen outcome definitions: "
    f"{len(valid_outcome_ids)}"
)

print(
    f"Frozen attribution rule IDs: "
    f"{len(valid_attribution_classes)}"
)

Input checksums passed: 8/8
Stage 6B gate: PASS_WITH_CAVEAT / passed_with_caveat
Controlled documents available: 56
Frozen strategy IDs: 9
Frozen outcome definitions: 18
Frozen attribution rule IDs: 9


## Documented Strategy Actions

Materialize only bounded, source-located actions that are explicitly documented. Company intentions, result explanations, and market-performance claims are not silently converted into strategy actions.


In [5]:
action_columns = [
    "action_id",
    "canonical_group",
    "reporting_entity",
    "brand_or_business",
    "strategy_code",
    "action_date",
    "action_period",
    "action_status",
    "action_description",
    "intended_mechanism",
    "geography",
    "ownership_scope",
    "source_document_id",
    "source_url",
    "source_locator",
    "claim_label",
    "extraction_status",
    "linkage_eligibility",
    "caveat",
]

action_specs = [
    # Wings Group
    (
        "S6CACT_WNG_001", "Wings Group", "WINGS Food", "GOLDA",
        "STR01", "2022-04-08", "2022", "implemented",
        "Launched the GOLDA Cappuccino ready-to-drink coffee variant.",
        "Expand the product offering within ready-to-drink coffee.",
        "Indonesia", "group portfolio; operating legal entity not inferred",
        "S6BDOC_WNG_005", "Official launch article, 8 April 2022",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "No effectiveness is inferred from the launch itself.",
    ),
    (
        "S6CACT_WNG_002", "Wings Group", "WINGS Food", "GOLDA",
        "STR02", "2022-04-08", "2022", "implemented",
        "Offered GOLDA Cappuccino in a 200 ml bottle at a stated price of Rp3,000.",
        "Use an explicit pack-price point to support affordability.",
        "Indonesia", "group portfolio; operating legal entity not inferred",
        "S6BDOC_WNG_005", "Official launch article, product availability section",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "The price point is an action attribute, not evidence of pricing effectiveness.",
    ),
    (
        "S6CACT_WNG_003", "Wings Group", "WINGS Food", "GOLDA",
        "STR04", "2022-04-08", "2022", "implemented",
        "Made GOLDA Cappuccino available through minimarkets, supermarkets and the Wings Official Store e-commerce channel.",
        "Broaden product availability across disclosed retail channels.",
        "Indonesia", "group portfolio; operating legal entity not inferred",
        "S6BDOC_WNG_005", "Official launch article, product availability section",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "Channel availability is not consumer reach.",
    ),
    (
        "S6CACT_WNG_004", "Wings Group", "Wings Care", "ProGuard",
        "STR01", "2022-08-18", "2022", "implemented",
        "Launched ProGuard antibacterial body wash.",
        "Enter or strengthen the antibacterial body-wash proposition with a differentiated formulation.",
        "Indonesia", "group portfolio; operating legal entity not inferred",
        "S6BDOC_WNG_004", "Official launch article, 18 August 2022",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "Health-performance claims are not independently evaluated in this project.",
    ),
    (
        "S6CACT_WNG_005", "Wings Group", "Wings Care", "ProGuard",
        "STR03", "2022-08-18", "2022", "implemented",
        "Supported the ProGuard launch with a campaign ambassador and the ProGuard Pro Festival activation.",
        "Build awareness and positioning around the product launch.",
        "Indonesia", "group portfolio; operating legal entity not inferred",
        "S6BDOC_WNG_004", "Official launch article, campaign and activation section",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "The existence of a campaign does not establish marketing effectiveness.",
    ),
    (
        "S6CACT_WNG_006", "Wings Group", "WINGS Food", "Ale-Ale",
        "STR01", "2023-03-23", "2023", "implemented",
        "Launched the Ale-Ale FunFlava sub-brand with Cocopandan as its first variant.",
        "Extend the flavored-drink portfolio with a new sub-brand and flavor proposition.",
        "Indonesia", "group portfolio; operating legal entity not inferred",
        "S6BDOC_WNG_006", "Official launch article, 23 March 2023",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "Any leadership language in the release remains company-reported.",
    ),
    (
        "S6CACT_WNG_007", "Wings Group", "WINGS Food", "Ale-Ale",
        "STR02", "2023-03-23", "2023", "implemented",
        "Offered Ale-Ale FunFlava Cocopandan at a stated price of Rp1,000 per cup.",
        "Maintain an explicit affordable price point for the launch.",
        "Indonesia", "group portfolio; operating legal entity not inferred",
        "S6BDOC_WNG_006", "Official launch article, price and availability section",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "The price point is not an outcome measure.",
    ),
    (
        "S6CACT_WNG_008", "Wings Group", "WINGS Food", "ISOPLUS",
        "STR01", "2023-04-02", "2023", "implemented",
        "Launched the ISOPLUS COCO hydration beverage variant.",
        "Extend the beverage portfolio with a coconut-water-based hydration proposition.",
        "Indonesia", "group portfolio; operating legal entity not inferred",
        "S6BDOC_WNG_007", "Official launch article, 2 April 2023",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "No performance impact is inferred from the launch.",
    ),

    # Mayora
    (
        "S6CACT_MYR_001", "Mayora", "PT Mayora Indah Tbk", "Consolidated portfolio",
        "STR01", "", "2025", "ongoing_documented",
        "Continued product innovation as an explicitly stated implemented strategy.",
        "Maintain product relevance and portfolio development.",
        "Indonesia plus export markets",
        "PT Mayora Indah Tbk and consolidated subsidiaries",
        "S6BDOC_MYR_AR_2025", "Annual Report 2025, Director's Report, strategy section",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "The report states the strategy at company level; brand-specific execution is not inferred.",
    ),
    (
        "S6CACT_MYR_002", "Mayora", "PT Mayora Indah Tbk", "Consolidated portfolio",
        "STR02", "", "2025", "ongoing_documented",
        "Maintained competitive selling prices without compromising stated quality standards.",
        "Balance affordability, demand and product positioning.",
        "Indonesia plus export markets",
        "PT Mayora Indah Tbk and consolidated subsidiaries",
        "S6BDOC_MYR_AR_2025", "Annual Report 2025, Director's Report, strategy section",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "This is a company-level pricing policy, not a quantified price/mix outcome.",
    ),
    (
        "S6CACT_MYR_003", "Mayora", "PT Mayora Indah Tbk", "Corporate capital allocation",
        "STR07", "2025-06-11", "2025", "implemented",
        "Implemented a share-buyback program approved with a maximum value of Rp1 trillion; 101.11 million shares had been repurchased by 31 December 2025.",
        "Execute a disclosed capital-allocation decision within the approved buyback framework.",
        "Indonesia",
        "PT Mayora Indah Tbk",
        "S6BDOC_MYR_AR_2025", "Annual Report 2025, Share Buyback section",
        "company_reported_action", "extracted",
        "context_only_for_consumer_performance",
        "A share buyback is a capital-allocation action, not evidence of consumer-market success.",
    ),
    (
        "S6CACT_MYR_004", "Mayora", "PT Mayora Indah Tbk", "Supply and procurement",
        "STR05", "", "2025", "ongoing_documented",
        "Reported maximizing the use of domestic raw materials as part of managing input and foreign-exchange risk.",
        "Reduce exposure to imported-input and exchange-rate volatility where feasible.",
        "Indonesia plus export operations",
        "PT Mayora Indah Tbk and consolidated subsidiaries",
        "S6BDOC_MYR_AR_2025", "Annual Report 2025, Risk Management section",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "No margin effect is inferred solely from this procurement policy.",
    ),

    # Unilever Indonesia — FY2024 annual report
    (
        "S6CACT_UNV_001", "Unilever Indonesia", "PT Unilever Indonesia Tbk", "Company portfolio",
        "STR04", "", "2024", "implemented",
        "Expanded focus on general trade, minimarkets and digital commerce while optimizing the distribution network.",
        "Increase product availability across growing channels and consumer touchpoints.",
        "Indonesia", "PT Unilever Indonesia Tbk controlled portfolio in 2024",
        "S6BDOC_UNV_AR_2024", "Annual Report 2024, Expanding Distribution Channels",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "Distribution actions are not automatically consumer reach.",
    ),
    (
        "S6CACT_UNV_002", "Unilever Indonesia", "PT Unilever Indonesia Tbk", "Company portfolio",
        "STR03", "", "2024", "implemented",
        "Standardized promotions as part of channel execution.",
        "Improve consistency of commercial execution across channels.",
        "Indonesia", "PT Unilever Indonesia Tbk controlled portfolio in 2024",
        "S6BDOC_UNV_AR_2024", "Annual Report 2024, Expanding Distribution Channels",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "No promotion-effect estimate is inferred.",
    ),
    (
        "S6CACT_UNV_003", "Unilever Indonesia", "PT Unilever Indonesia Tbk", "General Trade portfolio",
        "STR02", "", "2024", "implemented",
        "Used small-pack formats with coinage pricing in General Trade.",
        "Support affordability and accessibility for price-sensitive consumers.",
        "Indonesia", "PT Unilever Indonesia Tbk controlled portfolio in 2024",
        "S6BDOC_UNV_AR_2024", "Annual Report 2024, Expanding Distribution Channels",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "The reported effect is retained separately as a company-attribution claim.",
    ),
    (
        "S6CACT_UNV_004", "Unilever Indonesia", "PT Unilever Indonesia Tbk", "Wipol",
        "STR01", "", "2024", "implemented",
        "Relaunched Wipol with a stronger germ-and-virus-kill formula and a long-lasting fragrance proposition.",
        "Improve the product proposition in home hygiene.",
        "Indonesia", "Wipol controlled by PT Unilever Indonesia Tbk in 2024",
        "S6BDOC_UNV_AR_2024", "Annual Report 2024, Breakthrough Performance and Market Resilience",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "The source's leadership wording is not treated as an independently measured outcome.",
    ),
    (
        "S6CACT_UNV_005", "Unilever Indonesia", "PT Unilever Indonesia Tbk", "Trika",
        "STR01", "", "2024", "implemented",
        "Relaunched Trika with an improved and more competitive product mix.",
        "Broaden the proposition to serve more consumer needs.",
        "Indonesia", "Trika controlled by PT Unilever Indonesia Tbk in 2024",
        "S6BDOC_UNV_AR_2024", "Annual Report 2024, Breakthrough Performance and Market Resilience",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "No category-performance effect is inferred from the relaunch.",
    ),

    # Unilever Indonesia — Q1 2025 strategy update
    (
        "S6CACT_UNV_006", "Unilever Indonesia", "PT Unilever Indonesia Tbk", "Sunlight",
        "STR01", "", "Q1_2025", "implemented",
        "Used a superior-formulation and attractive-packaging proposition in the Sunlight strategy example.",
        "Strengthen product superiority and purchase intent.",
        "Indonesia", "Sunlight controlled by PT Unilever Indonesia Tbk in Q1 2025",
        "S6BDOC_UNV_008", "Q1 2025 Earnings Call, Strategy Update — Product and Packaging",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "Product claims remain company-reported.",
    ),
    (
        "S6CACT_UNV_007", "Unilever Indonesia", "PT Unilever Indonesia Tbk", "Sunlight",
        "STR03", "", "Q1_2025", "implemented",
        "Used social-first communication and large out-of-home promotion in the Sunlight launch approach.",
        "Increase launch visibility and communication reach.",
        "Indonesia", "Sunlight controlled by PT Unilever Indonesia Tbk in Q1 2025",
        "S6BDOC_UNV_008", "Q1 2025 Earnings Call, Strategy Update — Promotion",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "Media investment does not establish outcome causation.",
    ),
    (
        "S6CACT_UNV_008", "Unilever Indonesia", "PT Unilever Indonesia Tbk", "Sunlight",
        "STR04", "", "Q1_2025", "implemented",
        "Executed broad outlet coverage and visibility during the launch.",
        "Increase availability and visibility in directly served stores.",
        "Indonesia", "Sunlight controlled by PT Unilever Indonesia Tbk in Q1 2025",
        "S6BDOC_UNV_008", "Q1 2025 Earnings Call, Strategy Update — Place",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "The corresponding coverage measure is retained separately as OUT17, not consumer reach.",
    ),
    (
        "S6CACT_UNV_009", "Unilever Indonesia", "PT Unilever Indonesia Tbk", "Sunlight",
        "STR02", "", "Q1_2025", "implemented",
        "Applied competitive pricing through Net Revenue Management in the Sunlight strategy example.",
        "Manage price positioning while supporting the launch proposition.",
        "Indonesia", "Sunlight controlled by PT Unilever Indonesia Tbk in Q1 2025",
        "S6BDOC_UNV_008", "Q1 2025 Earnings Call, Strategy Update — Pricing",
        "company_reported_action", "extracted",
        "eligible_for_later_assessment",
        "No price/mix outcome is inferred without an explicitly reported measure.",
    ),
    (
        "S6CACT_UNV_010", "Unilever Indonesia", "PT Unilever Indonesia Tbk", "Ice Cream business",
        "STR09", "2025-12-08", "2025", "completed",
        "Completed the separation of the Ice Cream business on 8 December 2025.",
        "Change the controlled portfolio perimeter through a completed business separation.",
        "Indonesia", "controlled through 2025-12-08; separated thereafter",
        "S6BDOC_UNV_FS_2025", "FY2025 Financial Statements, discontinued-operation / Note 39 context",
        "company_reported_or_filing_action", "extracted",
        "context_only_for_post_disposal_comparison",
        "Later observations must respect the ownership end date and represented comparators.",
    ),
]

action_rows = []

for spec in action_specs:
    (
        action_id, canonical_group, reporting_entity, brand_or_business,
        strategy_code, action_date, action_period, action_status,
        action_description, intended_mechanism, geography, ownership_scope,
        source_document_id, source_locator, claim_label, extraction_status,
        linkage_eligibility, caveat,
    ) = spec

    if source_document_id not in valid_document_ids:
        raise RuntimeError(
            f"{action_id} references unknown document {source_document_id}."
        )

    action_rows.append(
        {
            "action_id": action_id,
            "canonical_group": canonical_group,
            "reporting_entity": reporting_entity,
            "brand_or_business": brand_or_business,
            "strategy_code": strategy_code,
            "action_date": action_date,
            "action_period": action_period,
            "action_status": action_status,
            "action_description": action_description,
            "intended_mechanism": intended_mechanism,
            "geography": geography,
            "ownership_scope": ownership_scope,
            "source_document_id": source_document_id,
            "source_url": resolved_source_url(source_document_id),
            "source_locator": source_locator,
            "claim_label": claim_label,
            "extraction_status": extraction_status,
            "linkage_eligibility": linkage_eligibility,
            "caveat": caveat,
        }
    )

stage6c_strategy_actions = pd.DataFrame(action_rows, columns=action_columns)

if len(stage6c_strategy_actions) != 22:
    raise RuntimeError(
        f"Expected 22 documented strategy actions, found {len(stage6c_strategy_actions)}."
    )

if stage6c_strategy_actions["action_id"].duplicated().any():
    raise RuntimeError("Duplicate Stage 6C action IDs detected.")

if not set(stage6c_strategy_actions["strategy_code"]).issubset(valid_strategy_codes):
    raise RuntimeError("An extracted action uses an unknown strategy code.")

print(f"Documented strategy actions: {len(stage6c_strategy_actions)}")
print(
    stage6c_strategy_actions.groupby(
        ["canonical_group", "strategy_code"]
    ).size()
)


Documented strategy actions: 22
canonical_group     strategy_code
Mayora              STR01            1
                    STR02            1
                    STR05            1
                    STR07            1
Unilever Indonesia  STR01            3
                    STR02            2
                    STR03            2
                    STR04            2
                    STR09            1
Wings Group         STR01            4
                    STR02            2
                    STR03            1
                    STR04            1
dtype: int64


## Observable Company-Result Observations

Materialize reported observations without imputing missing values or forcing unit conversion. Entity, geography, accounting scope, restatement status, and attribution class remain explicit on every row.


In [6]:
result_columns = [
    "result_id",
    "canonical_group",
    "reporting_entity",
    "reporting_scope",
    "geography",
    "reference_period",
    "outcome_id",
    "metric_name",
    "value_numeric",
    "unit",
    "value_qualifier",
    "value_status",
    "source_document_id",
    "source_url",
    "source_locator",
    "claim_label",
    "attribution_class",
    "evidence_status",
    "comparability_class",
    "caveat",
]

result_rows = []

def add_result(
    result_id,
    canonical_group,
    reporting_entity,
    reporting_scope,
    geography,
    reference_period,
    outcome_id,
    metric_name,
    value_numeric,
    unit,
    source_document_id,
    source_locator,
    claim_label,
    attribution_class,
    evidence_status,
    comparability_class,
    caveat,
    value_qualifier="exact_reported",
    value_status="reported",
):
    if source_document_id not in valid_document_ids:
        raise RuntimeError(
            f"{result_id} references unknown document {source_document_id}."
        )
    if outcome_id not in valid_outcome_ids:
        raise RuntimeError(
            f"{result_id} references unknown outcome {outcome_id}."
        )
    if attribution_class not in valid_attribution_classes:
        raise RuntimeError(
            f"{result_id} references unknown attribution class {attribution_class}."
        )

    result_rows.append(
        {
            "result_id": result_id,
            "canonical_group": canonical_group,
            "reporting_entity": reporting_entity,
            "reporting_scope": reporting_scope,
            "geography": geography,
            "reference_period": reference_period,
            "outcome_id": outcome_id,
            "metric_name": metric_name,
            "value_numeric": value_numeric,
            "unit": unit,
            "value_qualifier": value_qualifier,
            "value_status": value_status,
            "source_document_id": source_document_id,
            "source_url": resolved_source_url(source_document_id),
            "source_locator": source_locator,
            "claim_label": claim_label,
            "attribution_class": attribution_class,
            "evidence_status": evidence_status,
            "comparability_class": comparability_class,
            "caveat": caveat,
        }
    )


# ---------------------------------------------------------------------
# Indofood parent — consolidated group results
# ---------------------------------------------------------------------

for outcome_id, metric_name, value, unit in [
    ("OUT07", "net_sales", 110.83, "IDR_trillion"),
    ("OUT08", "sales_growth", 12.0, "percent"),
    ("OUT13", "operating_profit", 19.69, "IDR_trillion"),
    ("OUT14", "operating_margin", 17.8, "percent"),
]:
    add_result(
        f"S6CRES_IDF_2022_{outcome_id}",
        "Indofood",
        "PT Indofood Sukses Makmur Tbk",
        "parent_consolidated",
        "consolidated including businesses beyond primary packaged-FMCG scope",
        "FY2022",
        outcome_id,
        metric_name,
        value,
        unit,
        "S6BDOC_IDF_009",
        "FY2022 full-year results release",
        "company_reported",
        "ATTR04",
        "context_only",
        "context_only",
        "Parent consolidated results cannot be assigned to individual packaged-FMCG brands or treated as Indonesia-only consumer performance.",
    )

for outcome_id, metric_name, value, unit in [
    ("OUT07", "net_sales", 111.70, "IDR_trillion"),
    ("OUT13", "operating_profit", 19.66, "IDR_trillion"),
    ("OUT14", "operating_margin", 17.6, "percent"),
]:
    add_result(
        f"S6CRES_IDF_2023_{outcome_id}",
        "Indofood",
        "PT Indofood Sukses Makmur Tbk",
        "parent_consolidated_comparator_reported_in_FY2024_release",
        "consolidated including businesses beyond primary packaged-FMCG scope",
        "FY2023",
        outcome_id,
        metric_name,
        value,
        unit,
        "S6BDOC_IDF_011",
        "FY2024 full-year results release, prior-year comparator",
        "company_reported",
        "ATTR04",
        "context_only",
        "context_only",
        "FY2023 values are source-reported comparators in the FY2024 release; parent consolidated scope is broader than the primary FMCG portfolio.",
    )

for outcome_id, metric_name, value, unit in [
    ("OUT07", "net_sales", 115.79, "IDR_trillion"),
    ("OUT08", "sales_growth", 4.0, "percent"),
    ("OUT13", "operating_profit", 23.09, "IDR_trillion"),
    ("OUT14", "operating_margin", 19.9, "percent"),
]:
    add_result(
        f"S6CRES_IDF_2024_{outcome_id}",
        "Indofood",
        "PT Indofood Sukses Makmur Tbk",
        "parent_consolidated",
        "consolidated including businesses beyond primary packaged-FMCG scope",
        "FY2024",
        outcome_id,
        metric_name,
        value,
        unit,
        "S6BDOC_IDF_011",
        "FY2024 full-year results release",
        "company_reported",
        "ATTR04",
        "context_only",
        "context_only",
        "Parent consolidated results cannot be assigned to individual packaged-FMCG brands or segments without separate disclosure.",
    )

for outcome_id, metric_name, value, unit in [
    ("OUT07", "net_sales", 123.49, "IDR_trillion"),
    ("OUT08", "sales_growth", 7.0, "percent"),
    ("OUT13", "operating_profit", 24.57, "IDR_trillion"),
    ("OUT14", "operating_margin", 19.9, "percent"),
]:
    add_result(
        f"S6CRES_IDF_2025_{outcome_id}",
        "Indofood",
        "PT Indofood Sukses Makmur Tbk",
        "parent_consolidated_official_parent_republication",
        "consolidated including businesses beyond primary packaged-FMCG scope",
        "FY2025",
        outcome_id,
        metric_name,
        value,
        unit,
        "S6BDOC_IDF_012",
        "FY2025 full-year results republication, 30 March 2026",
        "company_reported",
        "ATTR04",
        "context_only",
        "context_only",
        "FY2025 is retained with an explicit source caveat because Stage 6B resolved an official parent/regulatory republication rather than an Indofood-owned-domain release.",
    )


# ---------------------------------------------------------------------
# Indofood CBP — consolidated ICBP results
# ---------------------------------------------------------------------

for outcome_id, metric_name, value, unit in [
    ("OUT07", "net_sales", 64.80, "IDR_trillion"),
    ("OUT08", "sales_growth", 14.0, "percent"),
    ("OUT13", "operating_profit", 13.38, "IDR_trillion"),
    ("OUT14", "operating_margin", 20.6, "percent"),
]:
    add_result(
        f"S6CRES_ICBP_2022_{outcome_id}",
        "Indofood",
        "PT Indofood CBP Sukses Makmur Tbk",
        "ICBP_consolidated",
        "Indonesia plus disclosed overseas operations",
        "FY2022",
        outcome_id,
        metric_name,
        value,
        unit,
        "S6BDOC_ICBP_009",
        "ICBP FY2022 full-year results release",
        "company_reported",
        "ATTR05",
        "context_only",
        "context_only",
        "ICBP consolidated results include overseas operations and are not equivalent to Indonesian household demand.",
    )

for outcome_id, metric_name, value, unit in [
    ("OUT07", "net_sales", 67.91, "IDR_trillion"),
    ("OUT08", "sales_growth", 5.0, "percent"),
    ("OUT13", "operating_profit", 14.39, "IDR_trillion"),
    ("OUT14", "operating_margin", 21.2, "percent"),
]:
    add_result(
        f"S6CRES_ICBP_2023_{outcome_id}",
        "Indofood",
        "PT Indofood CBP Sukses Makmur Tbk",
        "ICBP_consolidated",
        "Indonesia plus disclosed overseas operations",
        "FY2023",
        outcome_id,
        metric_name,
        value,
        unit,
        "S6BDOC_ICBP_010",
        "ICBP FY2023 full-year results release",
        "company_reported",
        "ATTR05",
        "context_only",
        "context_only",
        "ICBP consolidated results include overseas operations.",
    )

for outcome_id, metric_name, value, unit in [
    ("OUT07", "net_sales", 72.60, "IDR_trillion"),
    ("OUT08", "sales_growth", 7.0, "percent"),
    ("OUT13", "operating_profit", 16.32, "IDR_trillion"),
    ("OUT14", "operating_margin", 22.5, "percent"),
]:
    add_result(
        f"S6CRES_ICBP_2024_{outcome_id}",
        "Indofood",
        "PT Indofood CBP Sukses Makmur Tbk",
        "ICBP_consolidated",
        "Indonesia plus disclosed overseas operations",
        "FY2024",
        outcome_id,
        metric_name,
        value,
        unit,
        "S6BDOC_ICBP_011",
        "ICBP FY2024 full-year results release",
        "company_reported",
        "ATTR05",
        "context_only",
        "context_only",
        "ICBP consolidated results include overseas operations.",
    )

for outcome_id, metric_name, value, unit, locator in [
    ("OUT07", "net_sales", 74850923, "IDR_million", "FY2025 audited financial statements, Note 24 — Net Sales"),
    ("OUT13", "operating_profit", 16655530, "IDR_million", "FY2025 audited financial statements, Note 31 — Segment Information"),
    ("OUT15", "capital_expenditure_and_advances_for_fixed_assets", 3667975, "IDR_million", "FY2025 audited financial statements, Note 31 — Segment Information"),
]:
    add_result(
        f"S6CRES_ICBP_2025_{outcome_id}",
        "Indofood",
        "PT Indofood CBP Sukses Makmur Tbk",
        "ICBP_consolidated_audited",
        "Indonesia plus disclosed overseas operations",
        "FY2025",
        outcome_id,
        metric_name,
        value,
        unit,
        "S6BDOC_ICBP_FS_2025",
        locator,
        "company_reported_or_audited",
        "ATTR05",
        "context_only",
        "comparable_with_transformation",
        "FY2025 audited values retain their source-native million-Rupiah unit and consolidated ICBP geography; transformation is deferred.",
    )


# ---------------------------------------------------------------------
# Mayora — consolidated mixed-geography results
# ---------------------------------------------------------------------

mayora_values = {
    "FY2022": {
        "source": "S6BDOC_MYR_AR_2023",
        "locator": "Annual Report 2023, Financial Highlights",
        "values": [
            ("OUT07", "net_sales", 30669406, "IDR_million"),
            ("OUT11", "gross_profit", 6839423, "IDR_million"),
            ("OUT12", "gross_margin", 22.0, "percent"),
            ("OUT13", "operating_profit", 2433115, "IDR_million"),
            ("OUT14", "operating_margin", 8.0, "percent"),
        ],
    },
    "FY2023": {
        "source": "S6BDOC_MYR_AR_2023",
        "locator": "Annual Report 2023, Financial Highlights",
        "values": [
            ("OUT07", "net_sales", 31485008, "IDR_million"),
            ("OUT11", "gross_profit", 8407778, "IDR_million"),
            ("OUT12", "gross_margin", 27.0, "percent"),
            ("OUT13", "operating_profit", 4299475, "IDR_million"),
            ("OUT14", "operating_margin", 14.0, "percent"),
        ],
    },
    "FY2024": {
        "source": "S6BDOC_MYR_AR_2025",
        "locator": "Annual Report 2025, p. 6, Financial Highlights",
        "values": [
            ("OUT07", "net_sales", 36072949, "IDR_million"),
            ("OUT11", "gross_profit", 8302299, "IDR_million"),
            ("OUT12", "gross_margin", 23.0, "percent"),
            ("OUT13", "operating_profit", 3915365, "IDR_million"),
            ("OUT14", "operating_margin", 11.0, "percent"),
        ],
    },
    "FY2025": {
        "source": "S6BDOC_MYR_AR_2025",
        "locator": "Annual Report 2025, p. 6, Financial Highlights",
        "values": [
            ("OUT07", "net_sales", 38681562, "IDR_million"),
            ("OUT11", "gross_profit", 8492262, "IDR_million"),
            ("OUT12", "gross_margin", 22.0, "percent"),
            ("OUT13", "operating_profit", 3723712, "IDR_million"),
            ("OUT14", "operating_margin", 10.0, "percent"),
        ],
    },
}

for period, specification in mayora_values.items():
    for outcome_id, metric_name, value, unit in specification["values"]:
        add_result(
            f"S6CRES_MYR_{period[-4:]}_{outcome_id}",
            "Mayora",
            "PT Mayora Indah Tbk and consolidated subsidiaries",
            "consolidated",
            "Indonesia plus exports / international operations",
            period,
            outcome_id,
            metric_name,
            value,
            unit,
            specification["source"],
            specification["locator"],
            "company_reported",
            "ATTR05",
            "context_only",
            "comparable_with_caveat",
            "The consolidated result mixes domestic and export/international activity and cannot be interpreted as Indonesian household demand.",
        )


# ---------------------------------------------------------------------
# Unilever Indonesia — originally reported FY2022–FY2024 scope
# ---------------------------------------------------------------------

unilever_original = {
    "FY2022": [
        ("OUT07", "net_sales", 41219, "IDR_billion"),
        ("OUT11", "gross_profit", 19065, "IDR_billion"),
        ("OUT13", "operating_profit", 7069, "IDR_billion"),
    ],
    "FY2023": [
        ("OUT07", "net_sales", 38611, "IDR_billion"),
        ("OUT11", "gross_profit", 19195, "IDR_billion"),
        ("OUT13", "operating_profit", 6279, "IDR_billion"),
    ],
    "FY2024": [
        ("OUT07", "net_sales", 35139, "IDR_billion"),
        ("OUT11", "gross_profit", 16720, "IDR_billion"),
        ("OUT13", "operating_profit", 4415, "IDR_billion"),
    ],
}

for period, values in unilever_original.items():
    for outcome_id, metric_name, value, unit in values:
        add_result(
            f"S6CRES_UNV_ORIG_{period[-4:]}_{outcome_id}",
            "Unilever Indonesia",
            "PT Unilever Indonesia Tbk",
            "originally_reported_company_scope_in_2024_annual_report",
            "Indonesia operating company with export activity where reported",
            period,
            outcome_id,
            metric_name,
            value,
            unit,
            "S6BDOC_UNV_AR_2024",
            "Annual Report 2024, Financial Highlights",
            "company_reported",
            "ATTR03",
            "supported_with_caveat",
            "comparable_with_caveat",
            "These are the originally published historical values and are not overwritten by later represented continuing-operation comparators.",
        )


# ---------------------------------------------------------------------
# Unilever Indonesia — FY2025 audited continuing-operation presentation
# ---------------------------------------------------------------------

for period, values in {
    "FY2024_REPRESENTED": [
        ("OUT07", "net_sales_continuing_operation", 30622614, "IDR_million"),
        ("OUT11", "gross_profit_continuing_operation", 14559672, "IDR_million"),
        ("OUT13", "operating_profit_continuing_operation", 3807317, "IDR_million"),
    ],
    "FY2025": [
        ("OUT07", "net_sales_continuing_operation", 31943461, "IDR_million"),
        ("OUT11", "gross_profit_continuing_operation", 14997094, "IDR_million"),
        ("OUT13", "operating_profit_continuing_operation", 4592021, "IDR_million"),
    ],
}.items():
    for outcome_id, metric_name, value, unit in values:
        add_result(
            f"S6CRES_UNV_CONT_{period}_{outcome_id}",
            "Unilever Indonesia",
            "PT Unilever Indonesia Tbk",
            "continuing_operation_as_presented_in_FY2025_financial_statements",
            "Indonesia operating company with disclosed export activity",
            period,
            outcome_id,
            metric_name,
            value,
            unit,
            "S6BDOC_UNV_FS_2025",
            "FY2025 Financial Statements, Statement of Profit or Loss and Other Comprehensive Income, p. 2",
            "company_reported_or_audited",
            "ATTR03",
            "supported_with_caveat",
            "comparable_with_transformation",
            "The FY2024 comparator is explicitly restated and re-presented; it remains separate from the originally published FY2024 annual-report observation.",
        )

# Domestic net-sales observations from Note 24.
for period, value in [
    ("FY2024_REPRESENTED", 29813800),
    ("FY2025", 31001328),
]:
    add_result(
        f"S6CRES_UNV_DOM_{period}_OUT07",
        "Unilever Indonesia",
        "PT Unilever Indonesia Tbk",
        "continuing_operation_domestic_net_sales",
        "Indonesia",
        period,
        "OUT07",
        "domestic_net_sales",
        value,
        "IDR_million",
        "S6BDOC_UNV_FS_2025",
        "FY2025 Financial Statements, Note 24 — Net Sales by primary geographical market",
        "company_reported_or_audited",
        "ATTR03",
        "supported_with_caveat",
        "comparable_with_transformation",
        "Domestic company sales remain company-level across multiple categories and should not be assigned to an individual brand.",
    )

# FY2025 capital expenditure.
add_result(
    "S6CRES_UNV_2025_OUT15",
    "Unilever Indonesia",
    "PT Unilever Indonesia Tbk",
    "continuing_operation_segment_reporting",
    "Indonesia operating company",
    "FY2025",
    "OUT15",
    "capital_expenditure",
    1320686,
    "IDR_million",
    "S6BDOC_UNV_FS_2025",
    "FY2025 Financial Statements, Note 30 — Segment Information",
    "company_reported_or_audited",
    "ATTR03",
    "supported_with_caveat",
    "context_only",
    "Capital expenditure is an investment input and is not itself a performance score.",
)

# Q1 2025 launch distribution/availability measure.
add_result(
    "S6CRES_UNV_Q1_2025_OUT17",
    "Unilever Indonesia",
    "PT Unilever Indonesia Tbk",
    "Sunlight_launch_execution",
    "Indonesia",
    "Q1_2025",
    "OUT17",
    "direct_store_coverage_during_launch",
    70.0,
    "percent_direct_stores",
    "S6BDOC_UNV_008",
    "Q1 2025 Earnings Call, Strategy Update — Place",
    "company_reported",
    "ATTR01",
    "directly_supported",
    "context_only",
    "The source states more than 70% of direct stores; this is distribution availability, not consumer reach.",
    value_qualifier="lower_bound_reported_more_than",
)


stage6c_result_observations = pd.DataFrame(
    result_rows,
    columns=result_columns,
)

if len(stage6c_result_observations) != 69:
    raise RuntimeError(
        f"Expected 69 result observations, found {len(stage6c_result_observations)}."
    )

if stage6c_result_observations["result_id"].duplicated().any():
    raise RuntimeError("Duplicate Stage 6C result IDs detected.")

if not set(stage6c_result_observations["outcome_id"]).issubset(valid_outcome_ids):
    raise RuntimeError("An extracted result uses an unknown outcome ID.")

print(f"Observable result observations: {len(stage6c_result_observations)}")
print(
    stage6c_result_observations.groupby(
        ["canonical_group", "outcome_id"]
    ).size()
)


Observable result observations: 69
canonical_group     outcome_id
Indofood            OUT07         8
                    OUT08         6
                    OUT13         8
                    OUT14         7
                    OUT15         1
Mayora              OUT07         4
                    OUT11         4
                    OUT12         4
                    OUT13         4
                    OUT14         4
Unilever Indonesia  OUT07         7
                    OUT11         5
                    OUT13         5
                    OUT15         1
                    OUT17         1
dtype: int64


## Company-Reported Attribution Claims

Keep management explanations in a dedicated table. These rows capture what the company says contributed to a result; they are not independent causal findings and are never upgraded beyond `ATTR07 / company_reported` in Stage 6C.


In [7]:
claim_columns = [
    "claim_id",
    "canonical_group",
    "reporting_entity",
    "claim_period",
    "claimed_factor",
    "related_strategy_codes",
    "related_outcome_ids",
    "company_claim_summary",
    "source_document_id",
    "source_url",
    "source_locator",
    "attribution_class",
    "evidence_status",
    "required_treatment",
]

claim_specs = [
    (
        "S6CCLM_IDF_001", "Indofood", "PT Indofood Sukses Makmur Tbk",
        "FY2024", "vertically integrated operations and market position",
        "STR05", "OUT07;OUT13",
        "Management stated that the group's integrated operating model and market position supported growth and profitability.",
        "S6BDOC_IDF_011", "FY2024 full-year results release, CEO statement",
        "Treat as a company-reported explanation; the factor is broader than a discrete, isolated intervention.",
    ),
    (
        "S6CCLM_IDF_002", "Indofood", "PT Indofood Sukses Makmur Tbk",
        "FY2025", "vertically integrated business model",
        "STR05", "OUT07;OUT13",
        "Management stated that revenue and profitability growth were supported by the vertically integrated business model.",
        "S6BDOC_IDF_012", "FY2025 full-year results republication, CEO statement",
        "Retain the republication-source caveat and do not infer causation.",
    ),
    (
        "S6CCLM_ICBP_001", "Indofood", "PT Indofood CBP Sukses Makmur Tbk",
        "FY2024", "higher volume and enhanced productivity and efficiencies",
        "STR08", "OUT07;OUT13",
        "Management stated that the improved top line and EBIT were driven primarily by higher volume and enhanced productivity and efficiencies.",
        "S6BDOC_ICBP_011", "ICBP FY2024 full-year results release, CEO statement",
        "The source explicitly makes the attribution, but Stage 6C records it only as company-reported.",
    ),
    (
        "S6CCLM_MYR_001", "Mayora", "PT Mayora Indah Tbk",
        "FY2025", "raw-material price increases and resulting selling-price adjustment",
        "STR02", "OUT07",
        "The company stated that higher raw-material prices required selling-price adjustments and affected achievement of the 2025 sales target.",
        "S6BDOC_MYR_AR_2025", "Annual Report 2025, Comparison Between Actual Results and Targets",
        "Separate the documented pricing response from the company's explanation of the sales-target outcome.",
    ),
    (
        "S6CCLM_MYR_002", "Mayora", "PT Mayora Indah Tbk",
        "FY2025", "rising raw-material costs and higher cost of goods sold",
        "", "OUT13",
        "The company stated that rising raw-material costs increased cost of goods sold and were the main reason operating profit finished below target.",
        "S6BDOC_MYR_AR_2025", "Annual Report 2025, Comparison Between Actual Results and Targets",
        "This is an external/input-cost explanation rather than proof of a strategy effect.",
    ),
    (
        "S6CCLM_UNV_001", "Unilever Indonesia", "PT Unilever Indonesia Tbk",
        "FY2024", "small-pack formats with coinage pricing in General Trade",
        "STR02", "OUT09",
        "The company stated that its small-pack and coinage-pricing approach helped sustain sales volume and strengthen brand penetration in General Trade.",
        "S6BDOC_UNV_AR_2024", "Annual Report 2024, Expanding Distribution Channels",
        "Retain the stated volume effect as company-reported; brand-penetration wording is not converted into consumer reach or market share.",
    ),
]

claim_rows = []

for spec in claim_specs:
    (
        claim_id, canonical_group, reporting_entity, claim_period,
        claimed_factor, related_strategy_codes, related_outcome_ids,
        company_claim_summary, source_document_id, source_locator,
        required_treatment,
    ) = spec

    if source_document_id not in valid_document_ids:
        raise RuntimeError(
            f"{claim_id} references unknown document {source_document_id}."
        )

    for code_value in [
        value for value in related_strategy_codes.split(";") if value
    ]:
        if code_value not in valid_strategy_codes:
            raise RuntimeError(
                f"{claim_id} references unknown strategy code {code_value}."
            )

    for outcome_value in [
        value for value in related_outcome_ids.split(";") if value
    ]:
        if outcome_value not in valid_outcome_ids:
            raise RuntimeError(
                f"{claim_id} references unknown outcome ID {outcome_value}."
            )

    claim_rows.append(
        {
            "claim_id": claim_id,
            "canonical_group": canonical_group,
            "reporting_entity": reporting_entity,
            "claim_period": claim_period,
            "claimed_factor": claimed_factor,
            "related_strategy_codes": related_strategy_codes,
            "related_outcome_ids": related_outcome_ids,
            "company_claim_summary": company_claim_summary,
            "source_document_id": source_document_id,
            "source_url": resolved_source_url(source_document_id),
            "source_locator": source_locator,
            "attribution_class": "ATTR07",
            "evidence_status": "company_reported",
            "required_treatment": required_treatment,
        }
    )

stage6c_company_attribution_claims = pd.DataFrame(
    claim_rows,
    columns=claim_columns,
)

if len(stage6c_company_attribution_claims) != 6:
    raise RuntimeError(
        f"Expected 6 company-attribution claims, "
        f"found {len(stage6c_company_attribution_claims)}."
    )

if stage6c_company_attribution_claims["claim_id"].duplicated().any():
    raise RuntimeError("Duplicate Stage 6C claim IDs detected.")

if set(stage6c_company_attribution_claims["attribution_class"]) != {"ATTR07"}:
    raise RuntimeError("Company-attribution claims must remain ATTR07.")

if set(stage6c_company_attribution_claims["evidence_status"]) != {"company_reported"}:
    raise RuntimeError("Company-attribution claims were upgraded beyond company_reported.")

print(
    f"Company-reported attribution claims: "
    f"{len(stage6c_company_attribution_claims)}"
)
display(stage6c_company_attribution_claims)


Company-reported attribution claims: 6


,claim_id,canonical_group,reporting_entity,claim_period,claimed_factor,related_strategy_codes,related_outcome_ids,company_claim_summary,source_document_id,source_url,source_locator,attribution_class,evidence_status,required_treatment
0,S6CCLM_IDF_001,Indofood,PT Indofood Sukses Makmur Tbk,FY2024,vertically integrated operations and market position,STR05,OUT07;OUT13,Management stated that the group's integrated operating model and market position supported growth and profitability.,S6BDOC_IDF_011,https://www.indofood.com/menu/financial-press-releases/indofoods-full-year-financial-results-for-the-year-ended-31-december-2024,"FY2024 full-year results release, CEO statement",ATTR07,company_reported,"Treat as a company-reported explanation; the factor is broader than a discrete, isolated intervention."
1,S6CCLM_IDF_002,Indofood,PT Indofood Sukses Makmur Tbk,FY2025,vertically integrated business model,STR05,OUT07;OUT13,Management stated that revenue and profitability growth were supported by the vertically integrated business model.,S6BDOC_IDF_012,https://doc.irasia.com/listco/hk/firstpacific/press/p260330.pdf,"FY2025 full-year results republication, CEO statement",ATTR07,company_reported,Retain the republication-source caveat and do not infer causation.
2,S6CCLM_ICBP_001,Indofood,PT Indofood CBP Sukses Makmur Tbk,FY2024,higher volume and enhanced productivity and efficiencies,STR08,OUT07;OUT13,Management stated that the improved top line and EBIT were driven primarily by higher volume and enhanced productivity and efficiencies.,S6BDOC_ICBP_011,https://www.indofoodcbp.com/menu/financial-press-releases/icbps-full-year-financial-results-for-the-year-ended-31-december-2024,"ICBP FY2024 full-year results release, CEO statement",ATTR07,company_reported,"The source explicitly makes the attribution, but Stage 6C records it only as company-reported."
3,S6CCLM_MYR_001,Mayora,PT Mayora Indah Tbk,FY2025,raw-material price increases and resulting selling-price adjustment,STR02,OUT07,The company stated that higher raw-material prices required selling-price adjustments and affected achievement of the 2025 sales target.,S6BDOC_MYR_AR_2025,https://www.mayoraindah.co.id/assets/upload/file/ar2025-mayora-web-version.pdf,"Annual Report 2025, Comparison Between Actual Results and Targets",ATTR07,company_reported,Separate the documented pricing response from the company's explanation of the sales-target outcome.
4,S6CCLM_MYR_002,Mayora,PT Mayora Indah Tbk,FY2025,rising raw-material costs and higher cost of goods sold,,OUT13,The company stated that rising raw-material costs increased cost of goods sold and were the main reason operating profit finished below ...,S6BDOC_MYR_AR_2025,https://www.mayoraindah.co.id/assets/upload/file/ar2025-mayora-web-version.pdf,"Annual Report 2025, Comparison Between Actual Results and Targets",ATTR07,company_reported,This is an external/input-cost explanation rather than proof of a strategy effect.
5,S6CCLM_UNV_001,Unilever Indonesia,PT Unilever Indonesia Tbk,FY2024,small-pack formats with coinage pricing in General Trade,STR02,OUT09,The company stated that its small-pack and coinage-pricing approach helped sustain sales volume and strengthen brand penetration in Gene...,S6BDOC_UNV_AR_2024,https://www.unilever.co.id/files/indonesia-annual-reports-2024.pdf,"Annual Report 2024, Expanding Distribution Channels",ATTR07,company_reported,Retain the stated volume effect as company-reported; brand-penetration wording is not converted into consumer reach or market share.


## Extraction Exceptions and Boundaries

Register unresolved disclosure, scope, direct-link, metric-semantics, ownership and transformation issues. No exception is silently repaired through imputation, substitution, or post-hoc scope expansion.


In [8]:
exception_columns = [
    "exception_id",
    "canonical_group",
    "exception_type",
    "severity",
    "status",
    "affected_evidence",
    "description",
    "required_treatment",
]

exception_rows = [
    (
        "S6CEX001", "Wings Group", "annual_report_not_found",
        "caveat", "open", "company-result extraction",
        "No official Wings annual report was identified for FY2022–FY2025 in the controlled source discovery.",
        "Retain the disclosure limitation; do not infer zero or weak financial performance.",
    ),
    (
        "S6CEX002", "Wings Group", "audited_financials_not_found",
        "caveat", "open", "company-result extraction",
        "No official Wings audited financial statements were identified for FY2022–FY2025.",
        "Do not substitute unofficial estimates as equivalent audited evidence.",
    ),
    (
        "S6CEX003", "Wings Group", "market_share_claim_not_used",
        "caveat", "controlled", "OUT18",
        "Corporate launch material contains market-position language, but Stage 6C does not treat it as source-defined market share.",
        "Require a directly verified source that explicitly measures a defined market share before creating OUT18 observations.",
    ),
    (
        "S6CEX004", "Wings Group", "legal_entity_resolution_partial",
        "caveat", "open", "strategy actions",
        "Several Wings actions are documented at Wings Food or Wings Care level without a resolved operating legal entity.",
        "Retain group/business-unit attribution unless a later result linkage requires legal-entity precision.",
    ),
    (
        "S6CEX005", "Indofood", "parent_direct_document_partial",
        "minor", "open", "Indofood parent reporting",
        "Several Indofood annual-report and financial-statement documents remain official-index verified rather than directly resolved.",
        "Use period-result releases for the extracted values in Stage 6C and resolve file-level documents only when later analysis requires them.",
    ),
    (
        "S6CEX006", "Indofood", "parent_consolidated_scope",
        "caveat", "controlled", "Indofood parent result observations",
        "Indofood parent consolidated results include businesses beyond the primary packaged-FMCG analytical perimeter.",
        "Keep these observations context-only until a compatible segment-level result is available.",
    ),
    (
        "S6CEX007", "Indofood", "icbp_mixed_geography",
        "caveat", "controlled", "ICBP result observations",
        "ICBP consolidated reporting includes overseas operations and exports.",
        "Do not interpret consolidated ICBP results as Indonesian household demand.",
    ),
    (
        "S6CEX008", "Indofood", "strategy_action_specificity",
        "caveat", "open", "Indofood / ICBP strategy actions",
        "The reviewed result releases document operating factors and management explanations but do not provide sufficiently bounded actions for all strategy dimensions.",
        "Keep such statements in the company-attribution table rather than manufacturing discrete strategy actions.",
    ),
    (
        "S6CEX009", "Mayora", "mixed_geography",
        "caveat", "controlled", "Mayora result observations",
        "Mayora consolidated results mix domestic and export/international activity.",
        "Retain ATTR05/context-only treatment for Indonesian household-demand interpretation.",
    ),
    (
        "S6CEX010", "Mayora", "market_share_language_not_used",
        "caveat", "controlled", "OUT18",
        "Company materials discuss market position, but no Stage 6C OUT18 observation is created without a directly validated defined market-share measure.",
        "Keep market-share terminology source-specific and do not infer it from company narrative.",
    ),
    (
        "S6CEX011", "Mayora", "sensitivity_only_affiliate",
        "caveat", "controlled", "PT Tirta Fresindo Jaya / Le Minerale",
        "PT Tirta Fresindo Jaya / Le Minerale remains outside strict-control primary Mayora results.",
        "Do not add it to primary strategy or result datasets; retain only for separately labelled sensitivity analysis if later required.",
    ),
    (
        "S6CEX012", "Unilever Indonesia", "represented_comparator",
        "caveat", "controlled", "FY2024 results",
        "FY2024 originally published annual-report values and the FY2025 financial statements' restated/re-presented FY2024 continuing-operation comparator are not identical scopes.",
        "Preserve both observation sets with distinct result IDs and never overwrite the original FY2024 record.",
    ),
    (
        "S6CEX013", "Unilever Indonesia", "post_window_disposal",
        "caveat", "controlled", "SariWangi",
        "The SariWangi business sale completed on 2 March 2026, after the primary FY2022–FY2025 result window.",
        "Use the completion only as post-window ownership context and do not create a primary 2026 strategy-result bridge.",
    ),
    (
        "S6CEX014", "Cross-company context", "transformation_deferred",
        "informational", "deferred", "cross-company numeric comparison",
        "Extracted currency values remain in source-native million, billion or trillion Rupiah units and several reporting perimeters are incompatible.",
        "Perform only explicit, documented transformations in a later comparison stage after entity/geography/comparability gates are applied.",
    ),
]

stage6c_extraction_exceptions = pd.DataFrame(
    exception_rows,
    columns=exception_columns,
)

if len(stage6c_extraction_exceptions) != 14:
    raise RuntimeError(
        f"Expected 14 extraction exceptions, found {len(stage6c_extraction_exceptions)}."
    )

if stage6c_extraction_exceptions["exception_id"].duplicated().any():
    raise RuntimeError("Duplicate Stage 6C exception IDs detected.")

print(f"Extraction exceptions/caveats: {len(stage6c_extraction_exceptions)}")
print(stage6c_extraction_exceptions["severity"].value_counts())


Extraction exceptions/caveats: 14
severity
caveat           12
minor             1
informational     1
Name: count, dtype: int64


## Stage 6C Validation

Validate provenance, frozen taxonomies, separation of evidence classes, company/entity/geography scope, ownership periods, metric semantics, source-native units, missingness, and the prohibition on strategy-result inference.


In [9]:
validation_columns = [
    "check_id",
    "validation_area",
    "check_description",
    "result",
    "status",
    "critical_failure",
    "required_treatment",
]

all_source_ids = (
    set(stage6c_strategy_actions["source_document_id"])
    | set(stage6c_result_observations["source_document_id"])
    | set(stage6c_company_attribution_claims["source_document_id"])
)

used_strategy_codes = set(stage6c_strategy_actions["strategy_code"])
used_outcome_ids = set(stage6c_result_observations["outcome_id"])
used_attribution_classes = set(stage6c_result_observations["attribution_class"])

validation_rows = [
    (
        "S6C001", "input_integrity",
        "All eight governed Stage 5/6B inputs match their locked SHA-256 values.",
        "8/8 inputs passed", "passed", "no",
        "Stop Stage 6C if any inherited artifact differs.",
    ),
    (
        "S6C002", "prior_stage_gate",
        "Stage 6B final acquisition gate remains PASS_WITH_CAVEAT.",
        "PASS_WITH_CAVEAT", "passed_with_caveat", "no",
        "Carry all Stage 6B acquisition caveats into extraction.",
    ),
    (
        "S6C003", "document_registry",
        "The inherited controlled document registry retains 56 unique references.",
        f"{stage6b_documents['document_id'].nunique()} references",
        "passed", "no",
        "Do not create unregistered source identities during extraction.",
    ),
    (
        "S6C004", "strategy_taxonomy",
        "The frozen Stage 5 strategy taxonomy contains STR01–STR09.",
        "9 frozen codes", "passed", "no",
        "Reject post-hoc strategy dimensions.",
    ),
    (
        "S6C005", "outcome_taxonomy",
        "The frozen Stage 5 outcome taxonomy contains OUT01–OUT18.",
        "18 frozen outcomes", "passed", "no",
        "Do not create new outcome semantics after observing results.",
    ),
    (
        "S6C006", "attribution_taxonomy",
        "The frozen Stage 5 attribution taxonomy contains ATTR01–ATTR09.",
        "9 frozen classes", "passed", "no",
        "Use the pre-specified evidence-strength treatment.",
    ),
    (
        "S6C007", "action_identity",
        "Every documented strategy action has a unique action ID.",
        f"{stage6c_strategy_actions['action_id'].nunique()} unique actions",
        "passed", "no",
        "Do not duplicate one action across rows without a distinct strategy classification.",
    ),
    (
        "S6C008", "result_identity",
        "Every observable result has a unique result ID.",
        f"{stage6c_result_observations['result_id'].nunique()} unique results",
        "passed", "no",
        "Keep original and represented comparators as distinct observations.",
    ),
    (
        "S6C009", "claim_identity",
        "Every company-attribution claim has a unique claim ID.",
        f"{stage6c_company_attribution_claims['claim_id'].nunique()} unique claims",
        "passed", "no",
        "Do not merge distinct management explanations.",
    ),
    (
        "S6C010", "action_provenance",
        "Every strategy action maps to a registered Stage 6B document.",
        "complete", "passed", "no",
        "Reject actions without a controlled source reference.",
    ),
    (
        "S6C011", "result_provenance",
        "Every result observation maps to a registered Stage 6B document.",
        "complete", "passed", "no",
        "Reject observations without a controlled source reference.",
    ),
    (
        "S6C012", "claim_provenance",
        "Every company-attribution claim maps to a registered Stage 6B document.",
        "complete", "passed", "no",
        "Reject attribution claims without a controlled source reference.",
    ),
    (
        "S6C013", "source_urls",
        "All extracted evidence rows retain a resolved source URL.",
        "complete", "passed", "no",
        "Preserve direct or controlled-index provenance.",
    ),
    (
        "S6C014", "strategy_code_validity",
        "All action strategy codes belong to the frozen STR01–STR09 taxonomy.",
        f"{len(used_strategy_codes)} strategy codes used",
        "passed", "no",
        "Do not invent new action classes.",
    ),
    (
        "S6C015", "outcome_id_validity",
        "All result outcome IDs belong to the frozen OUT01–OUT18 taxonomy.",
        f"{len(used_outcome_ids)} outcome IDs used",
        "passed", "no",
        "Preserve outcome semantics.",
    ),
    (
        "S6C016", "attribution_class_validity",
        "All result attribution classes belong to the frozen ATTR01–ATTR09 taxonomy.",
        f"{len(used_attribution_classes)} attribution classes used",
        "passed", "no",
        "Apply scope-specific evidence treatment.",
    ),
    (
        "S6C017", "evidence_class_separation",
        "Strategy actions, observable results, and company-attribution claims are stored in separate canonical datasets.",
        "3 separated evidence classes", "passed", "no",
        "Do not collapse actions, outcomes and explanations into one score.",
    ),
    (
        "S6C018", "company_claim_boundary",
        "All company-attribution rows remain ATTR07 / company_reported.",
        "6/6 retained", "passed", "no",
        "Never upgrade a management explanation into independent causal evidence.",
    ),
    (
        "S6C019", "raw_storage",
        "Stage 6C creates no raw copyrighted source-document repository copies.",
        "none created", "passed", "no",
        "Keep corporate reports and webpages reference-only.",
    ),
    (
        "S6C020", "market_share_semantics",
        "No OUT18 market-share observation is created from corporate leadership language or survey-index terminology.",
        "0 OUT18 observations", "passed", "no",
        "Require a directly defined market-share measure before using market-share terminology.",
    ),
    (
        "S6C021", "consumer_reach_semantics",
        "Distribution/outlet coverage is not relabelled as consumer reach.",
        "OUT17 retained as distribution availability", "passed", "no",
        "Keep consumer reach and distribution availability distinct.",
    ),
    (
        "S6C022", "source_native_units",
        "Reported numeric values retain their source-native units and qualifiers.",
        "units retained", "passed_with_caveat", "no",
        "Perform cross-unit transformation only in a later governed stage.",
    ),
    (
        "S6C023", "missingness",
        "Unavailable or unextracted evidence is not filled with zero or silently imputed.",
        "preserved", "passed", "no",
        "Keep missing disclosure and extraction gaps explicit.",
    ),
    (
        "S6C024", "wings_disclosure",
        "Wings financial-disclosure gaps remain explicit exceptions and produce no fabricated financial observations.",
        "0 Wings financial-result rows", "passed_with_caveat", "no",
        "Do not interpret disclosure absence as weak performance.",
    ),
    (
        "S6C025", "wings_market_claims",
        "Wings corporate market-position language is excluded from OUT18.",
        "excluded", "passed", "no",
        "Use only an explicitly measured defined market share.",
    ),
    (
        "S6C026", "indofood_parent_scope",
        "Indofood parent consolidated results remain ATTR04/context-only for packaged-FMCG interpretation.",
        "retained", "passed_with_caveat", "no",
        "Do not assign parent results to individual brands or consumer segments.",
    ),
    (
        "S6C027", "icbp_geography",
        "ICBP consolidated results retain mixed-geography treatment.",
        "ATTR05/context-only", "passed_with_caveat", "no",
        "Do not interpret consolidated ICBP results as Indonesian household demand.",
    ),
    (
        "S6C028", "mayora_geography",
        "Mayora consolidated results retain mixed domestic/export treatment.",
        "ATTR05/context-only", "passed_with_caveat", "no",
        "Do not interpret consolidated Mayora results as Indonesian household demand.",
    ),
    (
        "S6C029", "mayora_sensitivity_scope",
        "PT Tirta Fresindo Jaya / Le Minerale does not appear in primary Stage 6C action or result datasets.",
        "excluded from primary extraction", "passed", "no",
        "Retain only for separately labelled sensitivity analysis if later required.",
    ),
    (
        "S6C030", "unilever_ownership",
        "Unilever Ice Cream separation is retained as a dated ownership action and SariWangi completion remains post-window context.",
        "ownership-period treatment retained", "passed_with_caveat", "no",
        "Do not carry disposed businesses beyond their valid ownership periods.",
    ),
    (
        "S6C031", "unilever_comparator_scope",
        "FY2024 originally published values and FY2024 restated/re-presented continuing-operation values remain separate.",
        "distinct observations retained", "passed_with_caveat", "no",
        "Never overwrite the original FY2024 observation with the later represented comparator.",
    ),
    (
        "S6C032", "interim_boundary",
        "The Q1 2025 distribution measure remains explicitly interim/launch context rather than a full-year result.",
        "retained", "passed", "no",
        "Require like-for-like treatment before any later interim comparison.",
    ),
    (
        "S6C033", "no_action_result_inference",
        "Stage 6C does not infer an outcome from a strategy action or a strategy from an outcome.",
        "none created", "passed", "no",
        "Reserve linkage assessment for the next governed stage.",
    ),
    (
        "S6C034", "no_causal_upgrade",
        "Temporal ordering and company attribution are not treated as causal identification.",
        "none upgraded", "passed", "no",
        "Use only the frozen evidence-status vocabulary in later linkage assessment.",
    ),
    (
        "S6C035", "no_composite_or_ranking",
        "Stage 6C creates no cross-company ranking, composite score or overall winner.",
        "none created", "passed", "no",
        "Keep extraction separate from synthesis.",
    ),
    (
        "S6C036", "reporting_boundary",
        "Stage 6C creates no analytical report or README.",
        "none created", "passed", "no",
        "Defer reporting until analysis, validation, visualization and findings are complete.",
    ),
    (
        "S6C037", "exception_registry",
        "Extraction caveats and unresolved boundaries are explicitly registered.",
        f"{len(stage6c_extraction_exceptions)} exceptions",
        "passed_with_caveat", "no",
        "Carry unresolved issues forward rather than silently repairing them.",
    ),
    (
        "S6C038", "stage_gate",
        "Governed evidence extraction is complete enough to begin strategy-result linkage eligibility screening.",
        "PASS_WITH_CAVEAT", "passed_with_caveat", "no",
        "Carry disclosure asymmetry, scope incompatibilities, represented comparators, legal boundaries and company-claim labels into Stage 6D.",
    ),
]

stage6c_extraction_validation = pd.DataFrame(
    validation_rows,
    columns=validation_columns,
)

expected_checks = {f"S6C{i:03d}" for i in range(1, 39)}

if set(stage6c_extraction_validation["check_id"]) != expected_checks:
    raise RuntimeError("Stage 6C validation registry is incomplete.")

if (stage6c_extraction_validation["critical_failure"] == "yes").any():
    raise RuntimeError("Stage 6C contains a critical validation failure.")

# Cross-table checks used by the validation claims.
for dataframe, id_column in [
    (stage6c_strategy_actions, "source_document_id"),
    (stage6c_result_observations, "source_document_id"),
    (stage6c_company_attribution_claims, "source_document_id"),
]:
    if not set(dataframe[id_column]).issubset(valid_document_ids):
        raise RuntimeError("An extracted record references an unregistered document.")

if (stage6c_strategy_actions["source_url"].str.strip() == "").any():
    raise RuntimeError("A strategy action is missing source provenance.")

if (stage6c_result_observations["source_url"].str.strip() == "").any():
    raise RuntimeError("A result observation is missing source provenance.")

if (stage6c_company_attribution_claims["source_url"].str.strip() == "").any():
    raise RuntimeError("A company-attribution claim is missing source provenance.")

if (stage6c_result_observations["outcome_id"] == "OUT18").any():
    raise RuntimeError("Unexpected OUT18 market-share observation detected.")

primary_text = " ".join(
    stage6c_strategy_actions["brand_or_business"].tolist()
    + stage6c_result_observations["reporting_entity"].tolist()
).lower()

if "le minerale" in primary_text or "tirta fresindo" in primary_text:
    raise RuntimeError(
        "Sensitivity-only Mayora affiliate leaked into primary Stage 6C datasets."
    )

final_stage6c = stage6c_extraction_validation.loc[
    stage6c_extraction_validation["check_id"] == "S6C038"
].iloc[0]

if (
    final_stage6c["result"] != "PASS_WITH_CAVEAT"
    or final_stage6c["status"] != "passed_with_caveat"
):
    raise RuntimeError("Unexpected Stage 6C final gate.")

print(f"Validation checks: {len(stage6c_extraction_validation)}")
print(
    "Critical failures: "
    f"{(stage6c_extraction_validation['critical_failure'] == 'yes').sum()}"
)
print(
    f"Final gate: {final_stage6c['result']} / {final_stage6c['status']}"
)


Validation checks: 38
Critical failures: 0
Final gate: PASS_WITH_CAVEAT / passed_with_caveat


## Canonical Stage 6C Outputs

Write the six canonical Stage 6C metadata/analytical outputs. Only derived factual tables are written; no raw reports, mirrored webpages, source images, README, or analytical report are created.


In [10]:
output_frames = {
    "metadata/stage6c_input_lock.csv": stage6c_input_lock,
    "data/analytical/stage6c_strategy_actions.csv": stage6c_strategy_actions,
    "data/analytical/stage6c_result_observations.csv": stage6c_result_observations,
    "data/analytical/stage6c_company_attribution_claims.csv":
        stage6c_company_attribution_claims,
    "metadata/stage6c_extraction_exceptions.csv": stage6c_extraction_exceptions,
    "metadata/stage6c_extraction_validation.csv": stage6c_extraction_validation,
}

for relative_path, dataframe in output_frames.items():
    destination = OUTPUT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(destination, index=False, encoding="utf-8")

print("Canonical Stage 6C outputs written:")
for relative_path, dataframe in output_frames.items():
    print(f"  {relative_path}: {len(dataframe)} rows")


Canonical Stage 6C outputs written:
  metadata/stage6c_input_lock.csv: 8 rows
  data/analytical/stage6c_strategy_actions.csv: 22 rows
  data/analytical/stage6c_result_observations.csv: 69 rows
  data/analytical/stage6c_company_attribution_claims.csv: 6 rows
  metadata/stage6c_extraction_exceptions.csv: 14 rows
  metadata/stage6c_extraction_validation.csv: 38 rows


## Output Manifest and Final Quality Assurance

Create SHA-256 checksums for the six canonical Stage 6C outputs, re-read every file, verify cardinalities and semantic boundaries, confirm that no prohibited raw/binary source material was created, and print the final Stage 6C gate.


In [11]:
manifest_rows = []

for relative_path, dataframe in output_frames.items():
    path = OUTPUT_ROOT / relative_path
    manifest_rows.append(
        {
            "file_path": relative_path,
            "artifact_type": "csv",
            "row_count": len(dataframe),
            "sha256": sha256_file(path),
            "locked_input_commit": INPUT_COMMIT,
        }
    )

stage6c_output_manifest = pd.DataFrame(manifest_rows)

manifest_path = OUTPUT_ROOT / "metadata/stage6c_output_manifest.csv"
manifest_path.parent.mkdir(parents=True, exist_ok=True)
stage6c_output_manifest.to_csv(
    manifest_path,
    index=False,
    encoding="utf-8",
)

expected_row_counts = {
    "metadata/stage6c_input_lock.csv": 8,
    "data/analytical/stage6c_strategy_actions.csv": 22,
    "data/analytical/stage6c_result_observations.csv": 69,
    "data/analytical/stage6c_company_attribution_claims.csv": 6,
    "metadata/stage6c_extraction_exceptions.csv": 14,
    "metadata/stage6c_extraction_validation.csv": 38,
}

for relative_path, expected_rows in expected_row_counts.items():
    reloaded = pd.read_csv(
        OUTPUT_ROOT / relative_path,
        dtype=str,
        keep_default_na=False,
    )
    if len(reloaded) != expected_rows:
        raise RuntimeError(
            f"{relative_path}: expected {expected_rows} rows, found {len(reloaded)}."
        )

manifest_reloaded = pd.read_csv(
    manifest_path,
    dtype=str,
    keep_default_na=False,
)

if len(manifest_reloaded) != 6:
    raise RuntimeError(
        f"Expected 6 Stage 6C manifest rows, found {len(manifest_reloaded)}."
    )

hash_failures = []

for row in manifest_reloaded.itertuples(index=False):
    actual_hash = sha256_file(OUTPUT_ROOT / row.file_path)
    if actual_hash != row.sha256:
        hash_failures.append(row.file_path)

if hash_failures:
    raise RuntimeError(
        f"Stage 6C output-manifest hash mismatch: {hash_failures}"
    )

all_runtime_files = [
    path for path in OUTPUT_ROOT.rglob("*") if path.is_file()
]

unexpected_binary_extensions = {
    ".pdf", ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx",
    ".jpg", ".jpeg", ".png", ".webp", ".zip",
}

unexpected_binaries = [
    str(path)
    for path in all_runtime_files
    if path.suffix.lower() in unexpected_binary_extensions
]

if unexpected_binaries:
    raise RuntimeError(
        "Unexpected raw/binary source files were created: "
        f"{unexpected_binaries}"
    )

if set(stage6c_company_attribution_claims["evidence_status"]) != {
    "company_reported"
}:
    raise RuntimeError("A company-attribution claim was improperly upgraded.")

if (stage6c_result_observations["outcome_id"] == "OUT18").any():
    raise RuntimeError("Market-share semantics boundary failed.")

print("Stage 6C final QA passed.")
print(f"Locked governed inputs: {len(stage6c_input_lock)}")
print(f"Documented strategy actions: {len(stage6c_strategy_actions)}")
print(f"Observable result observations: {len(stage6c_result_observations)}")
print(
    f"Company-reported attribution claims: "
    f"{len(stage6c_company_attribution_claims)}"
)
print(f"Extraction exceptions/caveats: {len(stage6c_extraction_exceptions)}")
print(f"Validation checks: {len(stage6c_extraction_validation)}")
print("Critical failures: 0")
print("Raw copyrighted source files written: 0")
print("OUT18 market-share observations created: 0")
print("Manifest hashes: 6/6 matched")
print(
    f"Stage 6C gate: "
    f"{final_stage6c['result']} / {final_stage6c['status']}"
)

display(
    stage6c_strategy_actions[
        [
            "action_id",
            "canonical_group",
            "strategy_code",
            "action_period",
            "brand_or_business",
        ]
    ]
)

display(
    stage6c_result_observations[
        [
            "result_id",
            "canonical_group",
            "reference_period",
            "outcome_id",
            "value_numeric",
            "unit",
            "attribution_class",
            "comparability_class",
        ]
    ]
)


Stage 6C final QA passed.
Locked governed inputs: 8
Documented strategy actions: 22
Observable result observations: 69
Company-reported attribution claims: 6
Extraction exceptions/caveats: 14
Validation checks: 38
Critical failures: 0
Raw copyrighted source files written: 0
OUT18 market-share observations created: 0
Manifest hashes: 6/6 matched
Stage 6C gate: PASS_WITH_CAVEAT / passed_with_caveat


,action_id,canonical_group,strategy_code,action_period,brand_or_business
0,S6CACT_WNG_001,Wings Group,STR01,2022,GOLDA
1,S6CACT_WNG_002,Wings Group,STR02,2022,GOLDA
2,S6CACT_WNG_003,Wings Group,STR04,2022,GOLDA
3,S6CACT_WNG_004,Wings Group,STR01,2022,ProGuard
4,S6CACT_WNG_005,Wings Group,STR03,2022,ProGuard
5,S6CACT_WNG_006,Wings Group,STR01,2023,Ale-Ale
6,S6CACT_WNG_007,Wings Group,STR02,2023,Ale-Ale
7,S6CACT_WNG_008,Wings Group,STR01,2023,ISOPLUS
8,S6CACT_MYR_001,Mayora,STR01,2025,Consolidated portfolio
9,S6CACT_MYR_002,Mayora,STR02,2025,Consolidated portfolio


,result_id,canonical_group,reference_period,outcome_id,value_numeric,unit,attribution_class,comparability_class
0,S6CRES_IDF_2022_OUT07,Indofood,FY2022,OUT07,110.83,IDR_trillion,ATTR04,context_only
1,S6CRES_IDF_2022_OUT08,Indofood,FY2022,OUT08,12.00,percent,ATTR04,context_only
2,S6CRES_IDF_2022_OUT13,Indofood,FY2022,OUT13,19.69,IDR_trillion,ATTR04,context_only
3,S6CRES_IDF_2022_OUT14,Indofood,FY2022,OUT14,17.80,percent,ATTR04,context_only
4,S6CRES_IDF_2023_OUT07,Indofood,FY2023,OUT07,111.70,IDR_trillion,ATTR04,context_only
...,...,...,...,...,...,...,...,...
64,S6CRES_UNV_CONT_FY2025_OUT13,Unilever Indonesia,FY2025,OUT13,4592021.00,IDR_million,ATTR03,comparable_with_transformation
65,S6CRES_UNV_DOM_FY2024_REPRESENTED_OUT07,Unilever Indonesia,FY2024_REPRESENTED,OUT07,29813800.00,IDR_million,ATTR03,comparable_with_transformation
66,S6CRES_UNV_DOM_FY2025_OUT07,Unilever Indonesia,FY2025,OUT07,31001328.00,IDR_million,ATTR03,comparable_with_transformation
67,S6CRES_UNV_2025_OUT15,Unilever Indonesia,FY2025,OUT15,1320686.00,IDR_million,ATTR03,context_only
